Qwen3 格式方案优化

In [5]:
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset, load_dataset

In [7]:
model_name = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.special_tokens_map

{'eos_token': '<|im_end|>',
 'pad_token': '<|endoftext|>',
 'additional_special_tokens': ['<|im_start|>',
  '<|im_end|>',
  '<|object_ref_start|>',
  '<|object_ref_end|>',
  '<|box_start|>',
  '<|box_end|>',
  '<|quad_start|>',
  '<|quad_end|>',
  '<|vision_start|>',
  '<|vision_end|>',
  '<|vision_pad|>',
  '<|image_pad|>',
  '<|video_pad|>']}

这个是 Llama3 的cht_template
```md
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
\n\n系统提示信息<|eot_id|>
<|start_header_id|>user<|end_header_id|>
\n\n用户输入<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
\n\n模型回答<|eot_id|>
```

```py
def process_func(example):
    MAX_LENGTH = 384    # Llama分词器会将一个中文字切分为多个token，因此需要放开一些最大长度，保证数据的完整性
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer(f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\n现在你要扮演皇帝身边的女人--甄嬛<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{example['instruction'] + example['input']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n", add_special_tokens=False)  # add_special_tokens 不在开头加 special_tokens
    response = tokenizer(f"{example['output']}<|eot_id|>", add_special_tokens=False)
    input_ids = instruction["input_ids"] + response["input_ids"] + [tokenizer.pad_token_id]
    attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]  # 因为eos token咱们也是要关注的所以 补充为1
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"] + [tokenizer.pad_token_id]  
    if len(input_ids) > MAX_LENGTH:  # 做一个截断
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }
```

In [12]:
text = [
    {
        "instruction": "小姐，别的秀女都在求中选，唯有咱们小姐想被撂牌子，菩萨一定记得真真儿的——",
        "input": "",
        "output": "嘘——都说许愿说破是不灵的。"
    },
    {
        "instruction": "这个温太医啊，也是古怪，谁不知太医不得皇命不能为皇族以外的人请脉诊病，他倒好，十天半月便往咱们府里跑。",
        "input": "",
        "output": "你们俩话太多了，我该和温太医要一剂药，好好治治你们。"
    },
    {
        "instruction": "嬛妹妹，刚刚我去府上请脉，听甄伯母说你来这里进香了。",
        "input": "",
        "output": "出来走走，也是散心。"
    },
    {
        "instruction": "嬛妹妹，我虽是一介御医，俸禄微薄，可是我保证会一生一世对你好，疼爱你，保护你，永远事事以你为重。本来没半月一次到府上去请脉，能够偶尔见一次妹妹的笑靥，已经心满意足了，可谁知——而且我也知道，妹妹心里是不愿意去殿选的。",
        "input": "",
        "output": "实初哥哥这么说，就枉顾我们一直以来的兄妹情谊了，嬛儿没有哥哥，一直把你当作自己的亲哥哥一样看待，自然相信哥哥会待妹妹好的——自然了，以后有了嫂子，你也会对嫂子更好。"
    },
    {
        "instruction": "实初虽然唐突了妹妹，却是真心实意地希望妹妹不要去应选，这不仅仅是因为我心里一直把妹妹当成……其实更是因为甄伯父曾经救过家父的性命。",
        "input": "",
        "output": "我们两家是世交，昔年恩义不过是父亲随手之劳，不必挂怀。"
    },
    {
        "instruction": "可是我父亲当年被诬，起因也是因为后宫争斗，不能独善其身。一介御医尚且如此，何况妹妹如果被选中的话，会身在其中啊。",
        "input": "",
        "output": "实初哥哥的话我都明白，只是我不去应选，迟早也是玉娆，家中无子，女儿还能不孝吗？"
    },
    {
        "instruction": "在京里休息了这些日子，早已经调养过来了。",
        "input": "",
        "output": "如今你住在自己京城的宅子里，不比从前住在外祖家，一墙之隔，见面也方便。"
    },
    {
        "instruction": "是啊。可是我总还想着我们一起长大的情分呢。诶？妹妹今日打扮得好生素净，可是细看起来还是个美人坯子，怎么看都是好的。",
        "input": "",
        "output": "沈大美人差矣，姐姐出落得这么标致，皇上见过必定会念念不忘。"
    },
    {
        "instruction": "你是谁？",
        "input": "",
        "output": "家父是大理寺少卿甄远道。"
    },
    {
        "instruction": "大理寺少卿，也不是什么高官。",
        "input": "",
        "output": "凡事不论官位高低，只论个理字。"
    },
    {
        "instruction": "你自负美貌，以为必然入选，便可以指使我吗？",
        "input": "",
        "output": "不敢，我只是为姐姐着想罢了。今日汉军旗大选，姐姐这样怕会惊动了圣驾，若是龙颜因此震怒，又岂是你我可以担当的？即便圣驾未惊，若传到他人耳中，坏了姐姐贤良的名声，更丢了咱们汉军旗的脸面。如此得不偿失，还望姐姐三思。"
    }]

In [9]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\n' + messages[0].content + '<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- set ns = namespace(multi_step_tool=true, last_query_index=messages|length - 1) %}
{%- for message in messages[::-1] %}
    {%- set index = (messages|length - 

In [14]:
example = text[0]
example

{'instruction': '小姐，别的秀女都在求中选，唯有咱们小姐想被撂牌子，菩萨一定记得真真儿的——',
 'input': '',
 'output': '嘘——都说许愿说破是不灵的。'}

In [23]:
tokenizer.apply_chat_template??

Signature:
tokenizer.apply_chat_template(
    conversation: Union[list[dict[str, str]], list[list[dict[str, str]]]],
    tools: Optional[list[Union[dict, Callable]]] = None,
    documents: Optional[list[dict[str, str]]] = None,
    chat_template: Optional[str] = None,
    add_generation_prompt: bool = False,
    continue_final_message: bool = False,
    tokenize: bool = True,
    padding: Union[bool, str, transformers.utils.generic.PaddingStrategy] = False,
    truncation: bool = False,
    max_length: Optional[int] = None,
    return_tensors: Union[str, transformers.utils.generic.TensorType, NoneType] = None,
    return_dict: bool = False,
    return_assistant_tokens_mask: bool = False,
    tokenizer_kwargs: Optional[dict[str, Any]] = None,
    **kwargs,
) -> Union[str, list[int], list[str], list[list[int]], transformers.tokenization_utils_base.BatchEncoding]
Source:   
    def apply_chat_template(
        self,
        conversation: Union[list[dict[str, str]], list[list[dict[str, str

In [28]:
MAX_LEN = 512

In [15]:
conversation = [
    {"role": "system", "content": "你是一个小说《甄嬛传》里的角色——甄嬛。"},
    {"role": "user", "content": "你是何人？"},
    {"role": "assistant", "content": "小女名甄嬛，家父大理寺少卿。"},
]

tokens = tokenizer.apply_chat_template(
    conversation,
    tokenize=False,
    add_generation_prompt=True,
)
tokens

'<|im_start|>system\n你是一个小说《甄嬛传》里的角色——甄嬛。<|im_end|>\n<|im_start|>user\n你是何人？<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n小女名甄嬛，家父大理寺少卿。<|im_end|>\n<|im_start|>assistant\n'

In [25]:
# 2. 生成完整token序列
full_input_ids = tokenizer.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=False,  # 不自动加assistant起始符
        return_tensors=None
    )

len(full_input_ids), tokenizer.decode(full_input_ids)

(50,
 '<|im_start|>system\n你是一个小说《甄嬛传》里的角色——甄嬛。<|im_end|>\n<|im_start|>user\n你是何人？<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n小女名甄嬛，家父大理寺少卿。<|im_end|>\n')

In [29]:
# 3. 生成到assistant开始前的前缀（用于mask掉）
prefix_input_ids = tokenizer.apply_chat_template(
    conversation[:-1],   # 只到user为止
    tokenize=True,
    add_generation_prompt=True    # 会自动加 "<|im_start|>assistant\n"
)

# 4. mask掉system+user部分
labels = full_input_ids.copy()
prefix_len = len(prefix_input_ids)
labels[:prefix_len] = [-100] * prefix_len

# 5. 截断
if len(full_input_ids) > MAX_LEN:
    full_input_ids = full_input_ids[:MAX_LEN]
    labels = labels[:MAX_LEN]

# 6. attention mask
attention_mask = [1] * len(full_input_ids)


In [30]:
full_input_ids, attention_mask, labels

([151644,
  8948,
  198,
  56568,
  101909,
  104032,
  26940,
  109628,
  123591,
  41683,
  25067,
  102073,
  100780,
  8545,
  109628,
  123591,
  1773,
  151645,
  198,
  151644,
  872,
  198,
  105043,
  98749,
  17340,
  11319,
  151645,
  198,
  151644,
  77091,
  198,
  151667,
  271,
  151668,
  271,
  30709,
  57750,
  13072,
  109628,
  123591,
  3837,
  45629,
  99910,
  103683,
  101277,
  82647,
  106987,
  1773,
  151645,
  198],
 [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [-100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  151667,
  271,
  151668,
  271,
  30709,
  57750,

In [32]:
tokenizer.decode(labels[prefix_len:])

'<think>\n\n</think>\n\n小女名甄嬛，家父大理寺少卿。<|im_end|>\n'

最终的可以写为：


In [33]:
def process_func(example):
    MAX_LEN = 4096

    # 1. 构造消息序列（符合Qwen模板结构）
    messages = [
        {"role": "system", "content": "你是皇帝身边的女人——甄嬛。"},
        {"role": "user", "content": example["instruction"] + example.get("input", "")},
        {"role": "assistant", "content": example["output"]}
    ]

    # 2. 生成完整token序列
    full_input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,  # 不自动加assistant起始符
        return_tensors=None
    )

    # 3. 生成到assistant开始前的前缀（用于mask掉）
    prefix_input_ids = tokenizer.apply_chat_template(
        messages[:-1],   # 只到user为止
        tokenize=True,
        add_generation_prompt=True    # 会自动加 "<|im_start|>assistant\n"
    )

    # 4. mask掉system+user部分
    labels = full_input_ids.copy()
    prefix_len = len(prefix_input_ids)
    labels[:prefix_len] = [-100] * prefix_len

    # 5. 截断
    if len(full_input_ids) > MAX_LEN:
        full_input_ids = full_input_ids[:MAX_LEN]
        labels = labels[:MAX_LEN]

    # 6. attention mask
    attention_mask = [1] * len(full_input_ids)

    return {
        "input_ids": full_input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }
